# Final Jewelry RAG retrieval evaluation

Evaluates the selected text and image retrieval methods against the unified gold dataset. Gold labels are used only after retrieval for scoring; all searches run against the full corpus.

## 1. Setup and load the unified gold dataset

In [1]:
from pathlib import Path
import importlib, json, os, sys, time, unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents, cwd / "ai-service"] if (p / "src" / "rag" / "scripts").is_dir() and (p / "rag_documents").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the Jewelry project or ai-service directory.")
RAG_SRC = PROJECT_ROOT / "src" / "rag"
if str(RAG_SRC) not in sys.path:
    sys.path.insert(0, str(RAG_SRC))

embedding_module = importlib.import_module("scripts.5_embeddings")
retrieval = importlib.import_module("scripts.7_retrieval")
image_embeddings = importlib.import_module("scripts.10_image_embeddings")
EVALUATION_DIR = PROJECT_ROOT / "data" / "rag" / "evaluations"
GOLD_FILE = EVALUATION_DIR / "final_retrieval_generation_gold.json"
RESULTS_FILE = EVALUATION_DIR / "final_retrieval_results.json"
gold = json.loads(GOLD_FILE.read_text(encoding="utf-8"))
CASES = gold["cases"]
TEXT_CASES = [case for case in CASES if "text" in case["modalities"]]
IMAGE_CASES = [case for case in CASES if "image" in case["modalities"]]
MODEL_NAME = "gemini-embedding-2"
IMAGE_MODEL_NAME = image_embeddings.DEFAULT_CONTEXT_MODEL
TOP_K = 5
EMBEDDING_CACHE = PROJECT_ROOT / "data" / "rag" / "embeddings"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "rag" / "processed_documents"
print(f"Gold cases: {len(CASES)} | text: {len(TEXT_CASES)} | image: {len(IMAGE_CASES)}")
print("Languages:", pd.Series([c["language"] for c in CASES]).value_counts().to_dict())
print("Expected PDFs in dataset:", gold.get("document_coverage"))
print("Gold labels are not used to filter retrieval candidates. Embeddings load from cache only.")


Gold cases: 193 | text: 100 | image: 93
Languages: {'en': 116, 'ar': 39, 'fr': 38}
Expected PDFs in dataset: {'expected_pdf_count': 39, 'text_labeled_pdf_count': 39, 'image_source_pdf_count': 36}
Gold labels are not used to filter retrieval candidates. Embeddings load from cache only.


## 2. Load cached text chunks and Gemini Embedding 2 vectors

In [2]:
chunk_records = []
for path in sorted(PROCESSED_ROOT.glob("*/chunks.json")):
    chunk_records.extend(json.loads(path.read_text(encoding="utf-8")).get("chunks", []))
chunks_by_id = pd.DataFrame(chunk_records).drop_duplicates("chunk_id").set_index("chunk_id")

# Match notebook 5's saved chunk order so the existing Gemini passage cache is reusable.
# This index contains the current source corpus; stale processed PDFs are not in this index.
chunk_order_file = EMBEDDING_CACHE / "BAAI__bge-m3" / "chunks.json"
if not chunk_order_file.is_file():
    raise FileNotFoundError("Saved chunk-order index missing; run notebook 4 first.")
chunk_ids = json.loads(chunk_order_file.read_text(encoding="utf-8"))["chunk_ids"]
missing_ids = [chunk_id for chunk_id in chunk_ids if chunk_id not in chunks_by_id.index]
if missing_ids:
    raise ValueError(f"Chunk-order index references missing chunks: {missing_ids[:5]}")
chunks = chunks_by_id.loc[chunk_ids].reset_index()

CURRENT_PDF_NAMES = {path.name for path in (PROJECT_ROOT / "rag_documents").rglob("*.pdf")}
chunks = chunks[chunks["file_name"].isin(CURRENT_PDF_NAMES)].reset_index(drop=True)
if chunks.empty:
    raise FileNotFoundError("No saved chunks found for current source PDFs. Run notebooks 1-4 first.")

# Cache-only loaders never call Gemini. A missing or mismatched cache raises an error.
chunk_pack = embedding_module.load_cached_gemini_embeddings(
    chunks["text"].astype(str).tolist(), text_type="passage", model_name=MODEL_NAME,
    cache_root=EMBEDDING_CACHE,
)
chunk_vectors = chunk_pack["embeddings"]
queries = [case["query"] for case in TEXT_CASES]
query_pack = embedding_module.load_cached_gemini_embeddings(
    queries, text_type="query", model_name=MODEL_NAME, cache_root=EMBEDDING_CACHE,
)
query_vectors = dict(zip(queries, query_pack["embeddings"]))

class CachedQueryModel:
    def encode(self, sentences, **kwargs):
        return np.asarray([query_vectors[str(sentence).strip()] for sentence in sentences], dtype=np.float32)

query_model = CachedQueryModel()
print("Current source PDFs:", len(CURRENT_PDF_NAMES), "documents:", chunks["file_name"].nunique(), "chunks:", len(chunks))
print("Loaded cached text passage and query vectors:", chunk_pack["cache_hit"], query_pack["cache_hit"])


Current source PDFs: 39 documents: 39 chunks: 283
Loaded cached text passage and query vectors: True True


## 3. Evaluate text retrieval against PDF and answer-keyword labels

In [3]:
def _norm(value):
    value = unicodedata.normalize("NFKD", str(value).casefold())
    return "".join(ch for ch in value if ch.isalnum())

def score_text_case(case):
    # Retrieval sees only the query and the full corpus. Gold labels enter below, for scoring.
    frame = retrieval.retrieve_dense(case["query"], query_model, chunk_vectors, chunks,
                                     top_k=TOP_K, model_name=MODEL_NAME)
    expected_docs = set(case.get("expected_documents", []))
    keywords = case.get("expected_keywords", [])
    doc_ranks = [int(row.rank) for row in frame.itertuples() if row.file_name in expected_docs]
    retrieved_doc_set = set(frame["file_name"].astype(str))
    retrieved_at_1 = set(frame.head(1)["file_name"].astype(str))
    expected_doc_coverage_at_1 = len(expected_docs & retrieved_at_1) / max(1, len(expected_docs))
    expected_doc_coverage_at_5 = len(expected_docs & retrieved_doc_set) / max(1, len(expected_docs))
    answer_ranks = []
    seen_keywords = set()
    for row in frame.itertuples():
        if row.file_name in expected_docs:
            body = _norm(row.text)
            matched = [kw for kw in keywords if _norm(kw) and _norm(kw) in body]
            if matched:
                answer_ranks.append(int(row.rank))
                seen_keywords.update(matched)
    answer_rank = min(answer_ranks) if answer_ranks else None
    return {
        "case_id": case["id"], "language": case["language"], "cross_pdf": case.get("cross_pdf", False),
        "query": case["query"], "expected_documents": sorted(expected_docs), "expected_keywords": keywords,
        "retrieved_documents": frame["file_name"].tolist(), "retrieved_pages": frame.get("page_number", pd.Series(dtype=int)).tolist(),
        "document_hit_at_1": bool(doc_ranks and min(doc_ranks) <= 1),
        "document_hit_at_5": bool(doc_ranks),
        "expected_document_coverage_at_1": expected_doc_coverage_at_1,
        "expected_document_coverage_at_5": expected_doc_coverage_at_5,
        "answer_recall_at_5": bool(answer_rank),
        "answer_rank": answer_rank, "keyword_coverage_at_5": len(seen_keywords) / max(1, len(keywords)),
        "retrieved_chunks": frame.to_dict(orient="records"),
    }

text_results = pd.DataFrame([score_text_case(case) for case in TEXT_CASES])
display(text_results[["case_id", "language", "document_recall_at_1", "document_recall_at_5", "answer_recall_at_5", "keyword_coverage_at_5"]].head())


,case_id,language,document_recall_at_1,document_recall_at_5,answer_recall_at_5,keyword_coverage_at_5
0,retrieval_001,en,True,True,True,1.0
1,retrieval_002,en,True,True,True,1.0
2,retrieval_003,en,True,True,True,1.0
3,retrieval_004,en,True,True,True,0.5
4,retrieval_005,en,True,True,True,1.0


## 4. Evaluate image retrieval against reviewed image-ID labels

In [4]:
records = image_embeddings.load_image_records(processed_root=PROCESSED_ROOT)
image_pack = image_embeddings.load_cached_image_embeddings(
    records,
    output_root=PROJECT_ROOT / "data" / "rag" / "embeddings" / "image_vlm_descriptions",
    context_model_name=IMAGE_MODEL_NAME,
)
image_vectors = image_pack["context_embeddings"]
image_query_vectors = {}
for case in IMAGE_CASES:
    cached_query = embedding_module.load_cached_gemini_embeddings(
        [case["query"]], text_type="query", model_name=IMAGE_MODEL_NAME,
        cache_root=PROJECT_ROOT / "data" / "rag" / "embeddings" / "image_query_embeddings",
    )
    image_query_vectors[case["query"]] = cached_query["embeddings"][0]

class CachedImageQueryModel:
    def encode(self, sentences, **kwargs):
        return np.asarray([image_query_vectors[str(sentence).strip()] for sentence in sentences], dtype=np.float32)

image_query_model = CachedImageQueryModel()
records_by_id = {str(record["image_id"]): record for record in records}
image_rows = []
for case in IMAGE_CASES:
    # Search the full image collection; gold image IDs are consulted only after retrieval.
    frame = retrieval.retrieve_images(
        case["query"], method="context_dense", model=image_query_model,
        model_name=IMAGE_MODEL_NAME, top_k=TOP_K,
        context_embeddings=image_vectors, image_records=records,
    )
    retrieved_ids = frame["image_id"].astype(str).tolist() if not frame.empty else []
    expected = set(case["expected_image_ids"])
    first_rank = next((rank for rank, image_id in enumerate(retrieved_ids, 1) if image_id in expected), None)
    hits = sum(image_id in expected for image_id in retrieved_ids)
    image_rows.append({
        "case_id": case["id"], "language": case["language"], "cross_pdf": case.get("cross_pdf", False),
        "query": case["query"], "expected_image_ids": sorted(expected), "retrieved_image_ids": retrieved_ids,
        "retrieved_files": frame["file_name"].tolist() if not frame.empty else [],
        "hit_at_1": first_rank == 1, "hit_at_5": first_rank is not None,
        "first_relevant_rank": first_rank, "recall_at_5": hits / max(1, len(expected)),
        "precision_at_5": hits / max(1, len(retrieved_ids)),
        "mrr_at_5": 1 / first_rank if first_rank and first_rank <= TOP_K else 0.0,
    })
image_results = pd.DataFrame(image_rows)
print("Image records:", len(records), "cached image embeddings:", image_pack["cache_hit"],
      "image query vectors loaded from cache:", len(image_query_vectors))
display(image_results[["case_id", "language", "hit_at_1", "hit_at_5", "recall_at_5", "mrr_at_5"]].head())


Image records: 307 cached image embeddings: True image query vectors loaded from cache: 93


,case_id,language,hit_at_1,hit_at_5,recall_at_5,mrr_at_5
0,image_003,en,True,True,1.000000,1.0
1,image_004,en,True,True,0.500000,1.0
2,image_005,en,False,False,0.000000,0.0
3,image_006,en,False,True,0.333333,0.5
4,image_007,en,True,True,1.000000,1.0


## 5. Review aggregate and language-specific metrics

In [ ]:
def summarize_text(frame):
    return {"cases":int(len(frame)),
        "DocumentHit@1":float(frame.document_hit_at_1.mean()),
        "DocumentHit@5":float(frame.document_hit_at_5.mean()),
        "ExpectedDocumentCoverage@1":float(frame.expected_document_coverage_at_1.mean()),
        "ExpectedDocumentCoverage@5":float(frame.expected_document_coverage_at_5.mean()),
        "AnswerRecall@5":float(frame.answer_recall_at_5.mean()),
        "KeywordCoverage@5":float(frame.keyword_coverage_at_5.mean())}
def summarize_image(frame):
    return {"cases":int(len(frame)),"Hit@1":float(frame.hit_at_1.mean()),"Hit@5":float(frame.hit_at_5.mean()),
        "Recall@5":float(frame.recall_at_5.mean()),"Precision@5":float(frame.precision_at_5.mean()),"MRR@5":float(frame.mrr_at_5.mean())}
text_overall=summarize_text(text_results)
image_overall=summarize_image(image_results)
text_by_language={k:summarize_text(v) for k,v in text_results.groupby("language")}
image_by_language={k:summarize_image(v) for k,v in image_results.groupby("language")}
text_cross=summarize_text(text_results[text_results.cross_pdf]) if text_results.cross_pdf.any() else None
image_cross=summarize_image(image_results[image_results.cross_pdf]) if image_results.cross_pdf.any() else None
summary={"text":text_overall,"text_by_language":text_by_language,"text_cross_pdf":text_cross,
         "image":image_overall,"image_by_language":image_by_language,"image_cross_pdf":image_cross}
print(json.dumps(summary,ensure_ascii=False,indent=2))


## 6. Save case-level results

In [ ]:
payload={"schema_version":1,"gold_dataset":str(GOLD_FILE.relative_to(PROJECT_ROOT)),
    "retrieval_configuration":{"text_method":"dense","text_embedding_model":MODEL_NAME,"image_method":"context_dense",
        "image_embedding_model":IMAGE_MODEL_NAME,"top_k":TOP_K,"candidate_filters":None},
    "summary":summary,"text_case_results":text_results.to_dict(orient="records"),
    "image_case_results":image_results.to_dict(orient="records")}
RESULTS_FILE.write_text(json.dumps(payload,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
print("Saved retrieval evaluation results:",RESULTS_FILE)
